In [ ]:
import json
from pathlib import Path

import exactextract
import geopandas as gpd
import plotly.express as px
import plotly.io as pio
import pycountry
from shapely.geometry import mapping

In [ ]:
# Folder with WIA in Sharepoint
base_dir = "./"
data_in_folder = base_dir + "data_in/"

# shapes subfolder
data_in_shapes = base_dir + "data_in/shapes/"

# out folder for maps
maps_out_folder = base_dir + "data_out/maps/PIN_population/"

# DBFS root
dbfs_base_dir = "./"
# WorldPop folder
wpop_dir = "WorldPop_rasters"
# indiv country rasters folder (constrained - total population)
country_pop_dir = "G2_CN_POP_R25A_100m"
# list of total_pop rasters available in DBFS
total_pop_rasters = [
    f.name
    for f in Path(
        data_in_folder + wpop_dir + "/" + country_pop_dir
    ).iterdir()
    if f.is_file()
]


In [ ]:
# define country/admin level/year
country = [
    # "Kenya",
    "Mali",
    # "Benin",
    # "Lebanon",
    # "Togo",
    # "Afghanistan",
    # "Ukraine",
    # "Burkina Faso",
    # "Niger",
    # "Honduras",
    # "El Salvador",
    # "Cameroon",
    # "Central African Republic",
    # "Myanmar",
    # "South Sudan",
    # "Syria",
    # "Ethiopia",
    # "Congo, The Democratic Republic of the",
    # "Haiti",
    # "Somalia",
    # "Sudan",
    # "Yemen",
    # "Saint Vincent and the Grenadines",
    # "Grenada",
]
country = country[0]
admin_level = "2"
year = "2025"

c_iso3 = (
    pycountry.countries.search_fuzzy(country)[0].alpha_3 if country != "Niger"
    # introduced Niger exception (known case to avoid Nigeria)
    else pycountry.countries.search_fuzzy(country)[1].alpha_3
)


In [ ]:
# identify raster id for the country
# NOTE: assumes raster already downloaded in DBFS
country_raster_id = [
    data_in_folder + wpop_dir + "/" + country_pop_dir + "/" + next(
        (
            f for f in total_pop_rasters
            if (c_iso3.lower() in f) and (f"_{year}_" in f)
        ),
        "None"
    )
]

# geojson output file name
geo_filename = f"geojson_{c_iso3}_adm{admin_level}"
# Get geojson in shapes
geojson_file = [
    f.name
    for f in Path(data_in_shapes).iterdir()
    if f.name.endswith(".zip") and geo_filename in f.name
]

# read country shape geojson into gdf
gdf = gpd.read_file(
    data_in_shapes + f"{geojson_file[0]}"
).to_crs('EPSG:4326')

# column pcode and name (assume harmonized across CODs)
pcode_col = f"adm{admin_level}_pcode"
name_col = f"adm{admin_level}_name"


In [ ]:
# compute exact extract
tot_pop_df = exactextract.exact_extract(
    country_raster_id,
    [
        {
            "geometry": mapping(g),
            "properties": {
                f"{pcode_col}": g_id,
                f"{name_col}": g_n,
            },
        } for g, g_id, g_n  in zip(
            gdf.geometry, gdf[pcode_col], gdf[name_col]
        )
    ],
    'sum',
    include_cols=[f"{pcode_col}", f"{name_col}"],
    output="pandas",
).rename(
    columns={
        "sum": f"tot_pop_{year}",
    }
).sort_values(pcode_col)


In [ ]:
# Check if there're none pop tiles for a pcode
# NOTE: exact extract may return null or zero if none pop tiles for a pcode
if len(tot_pop_df) != len(gdf):
    raise Exception("Stop execution: There're no pop tiles for pcode(s)")
elif tot_pop_df[f"tot_pop_{year}"].isnull().any():
    raise Exception("Stop execution: There're null pop values for pcode(s)")

In [ ]:
# save pop data
tot_pop_df.to_excel(
    maps_out_folder + f"{c_iso3}_adm{admin_level}_{year}_WorldPop.xlsx",
    index=False,
    sheet_name=f"{c_iso3}_adm{admin_level}_{year}",
)


In [ ]:
# Disable automatic rendering
pio.renderers.default = None

# plotly express choropleth map of country population by admin area polygon
fig_pop = px.choropleth(
    tot_pop_df,
    geojson=json.loads(gdf.to_json()),
    featureidkey=f"properties.{pcode_col}",
    locations=pcode_col,
    color=f"tot_pop_{year}",
    color_continuous_scale="YlOrRd",
    projection="mercator",
    width=1200,
    height=900,
)
fig_pop.update_geos(fitbounds="locations", visible=False)
fig_pop.update_layout(
    title=dict(
        text=f"WorldPop Population {year} ({c_iso3}_adm{admin_level})",
        x=0.5,  # x position (0-left, 1-right)
        y=0.97,  # y position (0-bottom, 1-top in normalized coordinates)
    ),
    margin={"r": 0, "t": 0, "l": 0, "b": 0}
)
fig_pop.update_coloraxes(colorbar_len=0.65)
fig_pop.update_traces(marker_line_color="Gainsboro", marker_line_width=0.5)


In [ ]:
# write figure as png direct locally
fig_pop.write_image(
    maps_out_folder + f"{c_iso3}_adm{admin_level}_{year}_WorldPop.png",
    format="png",
    scale=2,
)
